In [ ]:
# Mount Google Drive (for Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Not running in Google Colab - skipping drive mount")

# Set project root path
if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/face_based_attendance_system"
else:
    from pathlib import Path
    PROJECT_ROOT = str(Path("..").resolve())

print(f"📂 Project root: {PROJECT_ROOT}")

# 06 - Model Evaluation

This notebook covers comprehensive evaluation of the trained face recognition model.

## Evaluation Benchmarks:
1. LFW (Labeled Faces in the Wild)
2. CFP-FP (Celebrities in Frontal-Profile)
3. AgeDB-30 (Cross-Age Verification)
4. Custom Verification Testing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score
from scipy import interpolate
import pickle

print(f"PyTorch: {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 1. Load Trained Model

In [ ]:
import math

# MobileFaceNet definition (same as training)
class ECABlock(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        k = int(abs((math.log2(channels) + b) / gamma))
        k = k if k % 2 else k + 1
        k = max(3, k)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio, use_eca=True):
        super().__init__()
        hidden = in_ch * expand_ratio
        self.use_res = stride == 1 and in_ch == out_ch
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        layers += [nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        if use_eca:
            layers.append(ECABlock(hidden))
        layers += [nn.Conv2d(hidden, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        return x + self.conv(x) if self.use_res else self.conv(x)


class MobileFaceNet(nn.Module):
    def __init__(self, embedding_dim=512, use_eca=True):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 3, 2, 1, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, 3, 1, 1, groups=64, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        settings = [(2,64,5,2), (4,128,1,2), (2,128,6,1), (4,128,1,2), (2,128,2,1)]
        layers, in_ch = [], 64
        for exp, out, n, s in settings:
            for i in range(n):
                layers.append(InvertedResidual(in_ch, out, s if i==0 else 1, exp, use_eca))
                in_ch = out
        self.bottlenecks = nn.Sequential(*layers)
        self.conv3 = nn.Sequential(nn.Conv2d(128, 512, 1, bias=False), nn.BatchNorm2d(512), nn.PReLU(512))
        self.conv4 = nn.Sequential(nn.Conv2d(512, 512, 7, groups=512, bias=False), nn.BatchNorm2d(512))
        self.fc = nn.Linear(512, embedding_dim, bias=False)
        self.bn = nn.BatchNorm1d(embedding_dim)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bottlenecks(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x.flatten(1))
        x = self.bn(x)
        return F.normalize(x, p=2, dim=1)


def load_model(checkpoint_path, device):
    """Load trained model from checkpoint."""
    model = MobileFaceNet(embedding_dim=512, use_eca=True)
    
    if Path(checkpoint_path).exists():
        state_dict = torch.load(checkpoint_path, map_location=device)
        # Handle full checkpoint vs model-only
        if 'model_state_dict' in state_dict:
            state_dict = state_dict['model_state_dict']
        model.load_state_dict(state_dict)
        print(f"Loaded model from {checkpoint_path}")
    else:
        print(f"Warning: {checkpoint_path} not found, using random weights")
    
    model = model.to(device)
    model.eval()
    return model


# Load model
MODEL_PATH = '../models/checkpoints/mobilefacenet_final.pth'
model = load_model(MODEL_PATH, DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Evaluation Utilities

In [ ]:
from torchvision import transforms

# Preprocessing
preprocess = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


@torch.no_grad()
def get_embedding(model, image_path, device):
    """Extract embedding from a single image."""
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    embedding = model(tensor)
    return embedding.cpu().numpy().flatten()


@torch.no_grad()
def get_embeddings_batch(model, image_paths, device, batch_size=32):
    """Extract embeddings for multiple images."""
    embeddings = []
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_tensors = []
        
        for path in batch_paths:
            try:
                img = Image.open(path).convert('RGB')
                tensor = preprocess(img)
                batch_tensors.append(tensor)
            except Exception as e:
                print(f"Error loading {path}: {e}")
                batch_tensors.append(torch.zeros(3, 112, 112))
        
        batch = torch.stack(batch_tensors).to(device)
        emb = model(batch)
        embeddings.append(emb.cpu().numpy())
    
    return np.vstack(embeddings)


def cosine_similarity(emb1, emb2):
    """Compute cosine similarity between two embeddings."""
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))


print("Evaluation utilities defined.")

## 3. LFW Evaluation

In [ ]:
class LFWEvaluator:
    """
    Evaluate model on LFW (Labeled Faces in the Wild) benchmark.
    
    LFW structure:
        lfw_aligned/
            Person_Name/
                Person_Name_0001.jpg
                Person_Name_0002.jpg
        pairs.txt
    """
    def __init__(self, lfw_dir: str, pairs_file: str):
        self.lfw_dir = Path(lfw_dir)
        self.pairs = self._load_pairs(pairs_file)
    
    def _load_pairs(self, pairs_file: str):
        """Load LFW pairs from pairs.txt."""
        pairs = []
        
        with open(pairs_file, 'r') as f:
            lines = f.readlines()
        
        # Skip header
        for line in lines[1:]:
            parts = line.strip().split('\t')
            
            if len(parts) == 3:
                # Same person: name, img1_idx, img2_idx
                name = parts[0]
                idx1 = int(parts[1])
                idx2 = int(parts[2])
                path1 = self.lfw_dir / name / f"{name}_{idx1:04d}.jpg"
                path2 = self.lfw_dir / name / f"{name}_{idx2:04d}.jpg"
                pairs.append((path1, path2, 1))  # Same person
            
            elif len(parts) == 4:
                # Different persons: name1, idx1, name2, idx2
                name1, name2 = parts[0], parts[2]
                idx1, idx2 = int(parts[1]), int(parts[3])
                path1 = self.lfw_dir / name1 / f"{name1}_{idx1:04d}.jpg"
                path2 = self.lfw_dir / name2 / f"{name2}_{idx2:04d}.jpg"
                pairs.append((path1, path2, 0))  # Different persons
        
        print(f"Loaded {len(pairs)} pairs")
        return pairs
    
    @torch.no_grad()
    def evaluate(self, model, device, batch_size=32):
        """Evaluate model on LFW."""
        model.eval()
        
        similarities = []
        labels = []
        
        for path1, path2, label in tqdm(self.pairs, desc="LFW Evaluation"):
            if not path1.exists() or not path2.exists():
                continue
            
            emb1 = get_embedding(model, path1, device)
            emb2 = get_embedding(model, path2, device)
            
            sim = cosine_similarity(emb1, emb2)
            similarities.append(sim)
            labels.append(label)
        
        similarities = np.array(similarities)
        labels = np.array(labels)
        
        # Compute metrics
        return self._compute_metrics(similarities, labels)
    
    def _compute_metrics(self, similarities, labels):
        """Compute evaluation metrics."""
        # ROC curve
        fpr, tpr, thresholds = roc_curve(labels, similarities)
        roc_auc = auc(fpr, tpr)
        
        # Find optimal threshold
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
        
        # Accuracy at optimal threshold
        predictions = (similarities >= optimal_threshold).astype(int)
        accuracy = accuracy_score(labels, predictions)
        
        # TAR@FAR=0.001 (1:1000)
        far_target = 0.001
        tar_at_far = interpolate.interp1d(fpr, tpr)(far_target) if fpr.min() <= far_target <= fpr.max() else 0
        
        return {
            'accuracy': accuracy,
            'auc': roc_auc,
            'optimal_threshold': optimal_threshold,
            'tar_at_far_0001': float(tar_at_far),
            'fpr': fpr,
            'tpr': tpr
        }


# Example usage (uncomment with actual paths)
# lfw_evaluator = LFWEvaluator(
#     lfw_dir='../data/lfw_aligned',
#     pairs_file='../data/pairs.txt'
# )
# results = lfw_evaluator.evaluate(model, DEVICE)
# print(f"LFW Accuracy: {results['accuracy']*100:.2f}%")
# print(f"LFW AUC: {results['auc']:.4f}")

print("LFWEvaluator defined.")

## 4. CFP-FP Evaluation (Frontal-Profile)

In [ ]:
class CFPFPEvaluator:
    """
    Evaluate on CFP-FP (Celebrities in Frontal-Profile) benchmark.
    
    Tests model robustness to pose variation.
    """
    def __init__(self, cfp_dir: str):
        self.cfp_dir = Path(cfp_dir)
        self.pairs = self._load_pairs()
    
    def _load_pairs(self):
        """Load CFP-FP pairs from protocol files."""
        pairs = []
        protocol_dir = self.cfp_dir / 'Protocol' / 'Pair_list_P' / 'FP'
        
        # Load 10 folds
        for fold in range(1, 11):
            same_file = protocol_dir / f'{fold:02d}_same.txt'
            diff_file = protocol_dir / f'{fold:02d}_diff.txt'
            
            if same_file.exists():
                with open(same_file) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 2:
                            pairs.append((parts[0], parts[1], 1))
            
            if diff_file.exists():
                with open(diff_file) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 2:
                            pairs.append((parts[0], parts[1], 0))
        
        print(f"Loaded {len(pairs)} CFP-FP pairs")
        return pairs
    
    @torch.no_grad()
    def evaluate(self, model, device):
        """Evaluate on CFP-FP."""
        model.eval()
        
        similarities = []
        labels = []
        
        for path1, path2, label in tqdm(self.pairs, desc="CFP-FP Evaluation"):
            full_path1 = self.cfp_dir / 'Data' / 'Images' / path1
            full_path2 = self.cfp_dir / 'Data' / 'Images' / path2
            
            if not full_path1.exists() or not full_path2.exists():
                continue
            
            emb1 = get_embedding(model, full_path1, device)
            emb2 = get_embedding(model, full_path2, device)
            
            sim = cosine_similarity(emb1, emb2)
            similarities.append(sim)
            labels.append(label)
        
        similarities = np.array(similarities)
        labels = np.array(labels)
        
        # Compute accuracy
        fpr, tpr, thresholds = roc_curve(labels, similarities)
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
        predictions = (similarities >= optimal_threshold).astype(int)
        
        return {
            'accuracy': accuracy_score(labels, predictions),
            'auc': auc(fpr, tpr),
            'threshold': optimal_threshold
        }


print("CFPFPEvaluator defined.")

## 5. AgeDB-30 Evaluation (Cross-Age)

In [ ]:
class AgeDBEvaluator:
    """
    Evaluate on AgeDB-30 (Cross-Age) benchmark.
    
    Tests model robustness to age variation (30-year gap).
    """
    def __init__(self, agedb_bin_path: str):
        self.data = self._load_bin(agedb_bin_path)
    
    def _load_bin(self, bin_path: str):
        """Load AgeDB from binary file (InsightFace format)."""
        if not Path(bin_path).exists():
            print(f"AgeDB binary not found: {bin_path}")
            return None
        
        with open(bin_path, 'rb') as f:
            data = pickle.load(f, encoding='bytes')
        
        return data
    
    @torch.no_grad()
    def evaluate(self, model, device):
        """Evaluate on AgeDB-30."""
        if self.data is None:
            return {'accuracy': 0, 'error': 'Data not loaded'}
        
        model.eval()
        # Implementation depends on data format
        # Typically similar to LFW evaluation
        
        return {'accuracy': 0, 'note': 'Implementation needed'}


print("AgeDBEvaluator defined.")

## 6. Visualization

In [ ]:
def plot_roc_curves(results_dict: dict, title: str = "ROC Curves"):
    """
    Plot ROC curves for multiple benchmarks.
    
    Args:
        results_dict: {benchmark_name: {'fpr': fpr, 'tpr': tpr, 'auc': auc}}
    """
    plt.figure(figsize=(8, 8))
    
    for name, results in results_dict.items():
        if 'fpr' in results and 'tpr' in results:
            plt.plot(
                results['fpr'], 
                results['tpr'],
                label=f"{name} (AUC={results.get('auc', 0):.4f})"
            )
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_similarity_distribution(similarities, labels, title="Similarity Distribution"):
    """
    Plot similarity score distributions for same/different pairs.
    """
    same_sims = similarities[labels == 1]
    diff_sims = similarities[labels == 0]
    
    plt.figure(figsize=(10, 5))
    
    plt.hist(same_sims, bins=50, alpha=0.5, label='Same Person', density=True)
    plt.hist(diff_sims, bins=50, alpha=0.5, label='Different Person', density=True)
    
    plt.xlabel('Cosine Similarity')
    plt.ylabel('Density')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Visualization functions defined.")

## 7. Run Full Evaluation

In [ ]:
def run_full_evaluation(model, device):
    """
    Run evaluation on all benchmarks.
    """
    results = {}
    
    # LFW
    lfw_dir = Path('../data/lfw_aligned')
    pairs_file = Path('../data/pairs.txt')
    
    if lfw_dir.exists() and pairs_file.exists():
        print("\n=== LFW Evaluation ===")
        lfw_eval = LFWEvaluator(str(lfw_dir), str(pairs_file))
        results['LFW'] = lfw_eval.evaluate(model, device)
        print(f"LFW Accuracy: {results['LFW']['accuracy']*100:.2f}%")
        print(f"LFW AUC: {results['LFW']['auc']:.4f}")
    else:
        print("LFW dataset not found")
    
    # CFP-FP
    cfp_dir = Path('../data/cfp_fp')
    if cfp_dir.exists():
        print("\n=== CFP-FP Evaluation ===")
        cfp_eval = CFPFPEvaluator(str(cfp_dir))
        results['CFP-FP'] = cfp_eval.evaluate(model, device)
        print(f"CFP-FP Accuracy: {results['CFP-FP']['accuracy']*100:.2f}%")
    else:
        print("CFP-FP dataset not found")
    
    # Summary
    print("\n=== Evaluation Summary ===")
    for name, res in results.items():
        acc = res.get('accuracy', 0) * 100
        print(f"{name}: {acc:.2f}%")
    
    return results


# Run evaluation (uncomment when datasets are available)
# results = run_full_evaluation(model, DEVICE)

print("\nTo run evaluation:")
print("1. Download LFW aligned dataset")
print("2. Download pairs.txt")
print("3. Uncomment run_full_evaluation()")

## 8. Summary

### Benchmark Targets:

| Benchmark | Target | Description |
|-----------|--------|-------------|
| LFW | 98.5%+ | General face verification |
| CFP-FP | 95%+ | Frontal-profile verification |
| AgeDB-30 | 94%+ | Cross-age verification |

### Next Steps:
- Proceed to model export notebook
- Export to ONNX for deployment